# Module 3: Filtering -- Cleaning the Signal

This tutorial walks through the concepts step by step. Each section includes an explanation, an example, and an exercise.

## Step 1: Setup

Import libraries. We'll use `scipy.signal` for building proper digital filters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq
%matplotlib inline

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('Ready!')

**Exercise 1:** Try it yourself!

## Step 2: Clinical Context: Why We Filter

**Clinical scenario:** A neurology resident is reviewing an EEG recording. The tracing is a mess -- every channel shows a fast, jittery oscillation riding on top of the brain waves. It's 60 Hz power line noise. The attending says: "Apply a notch filter at 60 Hz." The noise vanishes. But what just happened? Did the filter remove *only* the noise?

Real EEG recordings contain:
- **2 Hz drift** -- slow baseline wander from electrode impedance changes
- **10 Hz alpha rhythm** -- the brain signal you actually want
- **60 Hz line noise** -- power line interference from hospital wiring
- **Random noise** -- thermal noise from electronics

Filtering separates what matters from what doesn't.

**The three basic filter types:**
- **High-pass**: removes low frequencies (removes drift)
- **Low-pass**: removes high frequencies (removes fast noise)
- **Band-pass**: keeps only a window of frequencies (standard EEG: 1-40 Hz)

**Exercise 2:** Try it yourself!

## Step 3: Building a Noisy Signal

Let's construct a realistic noisy EEG signal by adding together the components you'd see in a real recording. Then we'll take it apart with filters.

In [ ]:
np.random.seed(42)
fs = 1000  # Sampling rate
duration = 2.0
t = np.arange(0, duration, 1/fs)

# Individual components
drift = 30 * np.sin(2 * np.pi * 2 * t)         # 2 Hz slow drift
alpha = 20 * np.sin(2 * np.pi * 10 * t)        # 10 Hz alpha rhythm (the signal we want!)
line_noise = 12 * np.sin(2 * np.pi * 60 * t)   # 60 Hz power line noise
random_noise = 5 * np.random.randn(len(t))      # Random thermal noise

# The raw signal: everything mixed together
raw = drift + alpha + line_noise + random_noise

# Plot each component
fig, axes = plt.subplots(5, 1, figsize=(10, 10), sharex=True)

components = [
    (drift, '2 Hz Drift', '#c0392b'),
    (alpha, '10 Hz Alpha (brain signal)', '#2980b9'),
    (line_noise, '60 Hz Line Noise', '#e67e22'),
    (random_noise, 'Random Noise', '#888888'),
    (raw, 'RAW SIGNAL (all combined)', '#2c3e50'),
]

for ax, (y, name, color) in zip(axes, components):
    ax.plot(t, y, color=color, linewidth=0.8)
    ax.set_ylabel('uV')
    ax.set_title(name, fontsize=11, color=color)
    ax.set_ylim(-80, 80)

axes[-1].set_xlabel('Time (seconds)')
plt.suptitle('Anatomy of a Dirty EEG Signal', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Exercise 3:** Try it yourself!

## Step 4: FFT-Based Bandpass Filter (1-40 Hz)

The most direct way to filter: take the FFT, zero out the frequencies you don't want, then inverse FFT back to the time domain.

A **1-40 Hz bandpass** keeps only frequencies between 1 and 40 Hz -- removing the slow drift (<1 Hz) and the 60 Hz noise (>40 Hz). This is the standard clinical EEG display filter.

In [ ]:
# Compute the FFT
N = len(raw)
yf = fft(raw)
freqs = fftfreq(N, 1/fs)

# Create a bandpass mask: keep 1-40 Hz
low_cut = 1.0   # Hz
high_cut = 40.0  # Hz

mask = np.zeros(N)
# Keep frequencies between low_cut and high_cut (positive and negative)
mask[(np.abs(freqs) >= low_cut) & (np.abs(freqs) <= high_cut)] = 1.0

# Apply the mask and inverse FFT
yf_filtered = yf * mask
filtered = np.real(np.fft.ifft(yf_filtered))

# Plot before and after
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

axes[0].plot(t, raw, color='#2c3e50', linewidth=0.8)
axes[0].set_title('Raw Signal (drift + alpha + 60 Hz + noise)', fontsize=11)
axes[0].set_ylabel('uV')
axes[0].set_ylim(-80, 80)

axes[1].plot(t, filtered, color='#27ae60', linewidth=1.2)
axes[1].set_title('After 1-40 Hz Bandpass Filter (FFT method)', fontsize=11, color='#27ae60')
axes[1].set_ylabel('uV')
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylim(-80, 80)

plt.tight_layout()
plt.show()

print('The drift and 60 Hz noise are gone. The 10 Hz alpha rhythm is clearly visible.')

**Exercise 4:** Try it yourself!

## Step 5: Notch Filter at 60 Hz

Sometimes you want to remove a **single specific frequency** rather than a range. A **notch filter** cuts out a narrow band around one frequency. The classic use: removing 60 Hz power line noise (50 Hz in Europe).

We'll use `scipy.signal.iirnotch` which designs a proper IIR notch filter.

In [ ]:
# Design a notch filter at 60 Hz
f_notch = 60.0   # Frequency to remove
Q = 30.0         # Quality factor (higher = narrower notch)

b_notch, a_notch = signal.iirnotch(f_notch, Q, fs)

# Create a signal with alpha + 60 Hz noise (no drift for clarity)
clean_alpha = 25 * np.sin(2 * np.pi * 10 * t)
noisy_signal = clean_alpha + 15 * np.sin(2 * np.pi * 60 * t)

# Apply the notch filter
notch_filtered = signal.filtfilt(b_notch, a_notch, noisy_signal)

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

axes[0].plot(t, noisy_signal, color='#c0392b', linewidth=0.8)
axes[0].set_title('Alpha + 60 Hz Noise (before notch)', fontsize=11, color='#c0392b')
axes[0].set_ylabel('uV')

axes[1].plot(t, notch_filtered, color='#27ae60', linewidth=1.2)
axes[1].set_title('After 60 Hz Notch Filter -- only alpha remains', fontsize=11, color='#27ae60')
axes[1].set_ylabel('uV')
axes[1].set_xlabel('Time (seconds)')

plt.tight_layout()
plt.show()

**Exercise 5:** Try it yourself!

## Step 6: Frequency Spectrum Before and After Filtering

Let's look at the filtering from the frequency domain perspective. The FFT power spectrum reveals exactly what the filter removed.

In [ ]:
def plot_spectrum(signal_data, fs, label, color, ax):
    """Plot single-sided amplitude spectrum."""
    N = len(signal_data)
    yf = np.abs(fft(signal_data))[:N//2] * 2 / N
    freqs = fftfreq(N, 1/fs)[:N//2]
    ax.plot(freqs, yf, color=color, linewidth=1.5, label=label)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Amplitude')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Before filtering
plot_spectrum(raw, fs, 'Raw signal', '#c0392b', axes[0])
axes[0].set_title('Before Filtering', fontsize=12)
axes[0].set_xlim(0, 80)
axes[0].axvline(2, color='gray', linestyle=':', alpha=0.5, label='2 Hz drift')
axes[0].axvline(10, color='#2980b9', linestyle=':', alpha=0.5, label='10 Hz alpha')
axes[0].axvline(60, color='#e67e22', linestyle=':', alpha=0.5, label='60 Hz noise')
axes[0].legend(fontsize=8)

# After bandpass filtering
plot_spectrum(filtered, fs, 'Bandpass 1-40 Hz', '#27ae60', axes[1])
axes[1].set_title('After 1-40 Hz Bandpass', fontsize=12)
axes[1].set_xlim(0, 80)
axes[1].axvline(10, color='#2980b9', linestyle=':', alpha=0.5, label='10 Hz alpha')
axes[1].axvspan(1, 40, alpha=0.1, color='green', label='Passband')
axes[1].legend(fontsize=8)

plt.suptitle('Power Spectrum: Before vs After Filtering', fontsize=13)
plt.tight_layout()
plt.show()

print('Left: peaks at 2 Hz (drift), 10 Hz (alpha), and 60 Hz (noise).')
print('Right: only the 10 Hz alpha peak survives the 1-40 Hz bandpass.')

**Exercise 6:** Try it yourself!

## Step 7: Filter Rules of Thumb for Clinical EEG

Here's what you'll see on most EEG machines:

| Setting | Typical Value | What It Does |
|---|---|---|
| High-pass | 0.5-1 Hz | Removes electrode drift |
| Low-pass | 35-70 Hz | Removes muscle artifact and high-freq noise |
| Notch | 60 Hz (US) / 50 Hz (EU) | Removes power line interference |
| Standard display | 1-40 Hz bandpass | Default clinical view |

**Key tradeoff:** Every filter removes both noise AND signal at that frequency. A notch at 60 Hz also removes gamma brain activity at 60 Hz. Too aggressive a high-pass can clip slow seizure patterns. The attending adjusts filters, re-reads, adjusts again -- it's an interactive process.

**Exercise 7:** Try it yourself!

## Step 8: Exercise: Extract Alpha Band

Apply a bandpass filter to keep only the alpha band (8-13 Hz).

**Exercise 8:** Try it yourself!

In [ ]:
# YOUR CODE HERE
# Task: Apply a bandpass filter to keep only 8-13 Hz (the alpha band).
#
# Use the FFT-based approach:
# 1. Compute FFT of 'raw' signal: yf = fft(raw)
# 2. Get frequencies: freqs = fftfreq(len(raw), 1/fs)
# 3. Create mask: keep only 8-13 Hz
#    mask = np.zeros(len(raw))
#    mask[(np.abs(freqs) >= 8) & (np.abs(freqs) <= 13)] = 1.0
# 4. Apply and inverse FFT: alpha_only = np.real(np.fft.ifft(yf * mask))
# 5. Plot the result and compare to the original alpha component


## Step 9: Exercise: What Happens with a Too-Narrow Filter?

Explore the consequences of over-filtering. What happens to the signal if you use a very narrow bandpass like 9-11 Hz instead of 8-13 Hz?

**Exercise 9:** Try it yourself!

In [ ]:
# YOUR CODE HERE
# Task: Apply a narrow 9-11 Hz bandpass and a wider 8-13 Hz bandpass.
# Plot both results side by side. What's different?
#
# Try both FFT-based approach and scipy butterworth:
# For scipy approach:
#   sos = signal.butter(4, [9, 11], btype='band', fs=fs, output='sos')
#   narrow = signal.sosfiltfilt(sos, raw)
#
#   sos = signal.butter(4, [8, 13], btype='band', fs=fs, output='sos')
#   wide = signal.sosfiltfilt(sos, raw)
#
# Question: Is the narrow filter better or worse? Why?
